# 생활법률 AI 어시스턴트: 법률 PDF 기반 Corrective RAG + 답변 평가 + HITL

## 이 노트북의 역할
법률 PDF 3종(개인정보 보호법, 근로기준법, 주택임대차보호법)을 **조항 단위로 쪼개 벡터DB(FAISS)에 저장**하고, 질문을 분석해 알맞은 **전문가 에이전트**(법률별 + 웹 검색)를 고른 뒤, 답변을 **평가하고 사람이 승인**하는 전체 파이프라인을 만듭니다. 지금까지 배운 StateGraph · 조건부 엣지 · 구조화 출력 · 체크포인터 · HITL 이 모두 쓰이는 종합 실습입니다.

| 단계 | 내용 | 배우는 것 |
|---|---|---|
| 1. 환경 설정 | 키, `get_llm()`, 라이브러리 | 제공자 선택 |
| 2. 법률 문서 인덱싱 | PDF → 장/조 단위 분할 → Document → FAISS | 정규식 파싱, 메타데이터, 인덱스 재사용 |
| 3. 도구 정의 | 법률 검색 3종(+Re-rank), 웹 검색 | `ContextualCompressionRetriever`, Cross-Encoder |
| 4-1. Corrective RAG 에이전트 | 검색 → 정보 추출·평가 → (부족하면) 질문 재작성 → 답변 | 구조화 출력, 루프, 서브그래프 |
| 4-2. 질문 라우팅 | 질문에 맞는 전문가를 1개 이상 선택해 **병렬** 실행 | Adaptive RAG, 병렬 분기, 커스텀 reducer |
| 5. 답변 평가 + HITL | 평가 에이전트의 점수를 보고 사람이 승인/거절 | 평가 노드, 사람 승인 루프 |
| 6. Gradio 챗봇 | `interrupt_before` 로 중단 → 웹 UI 에서 승인 | 세션별 `thread_id`, 상태 수정 후 재개 |

## 전체 흐름
```text
질문 → analyze_question(라우터) ─┬ search_personal ─┐   각 search_* = Corrective RAG 에이전트 (4-1)
                                ├ search_labor    ─┤
                                ├ search_housing  ─┼→ generate_answer → evaluate_answer → 사람 승인 ─ 승인 → END
                                ├ search_web      ─┘                                        └ 거절 → analyze_question
                                └ llm_fallback ───────────────────────────────→ END   (법률·검색과 무관한 질문)

Corrective RAG 에이전트:  retrieve → extract_and_evaluate ─ 정보 충분 ──→ generate_answer → END
                                  ↑                        └ 부족 → rewrite_query ┘ (최대 2회)
```

## 사용 모델 (`get_llm()`)
- `PROVIDER` 값(`"groq"` / `"openai"` / `"upstage"`, 기본값 `"upstage"`)만 바꾸면 이후 모든 셀의 LLM 이 바뀝니다. 바꾼 뒤에는 LLM 설정 셀부터 다시 실행하세요. (에이전트를 만든 셀은 다시 실행해야 새 모델이 반영됩니다)
- 구조화 출력(`with_structured_output`)을 많이 쓰므로 모델에 따라 결과가 달라질 수 있습니다.
- **임베딩은 항상 Upstage** (`solar-embedding-1-large`) 입니다. 벡터DB 를 만든 모델과 검색 모델이 같아야 하므로 어떤 제공자를 골라도 `UPSTAGE_API_KEY` 는 필요합니다.

## 사전 준비 · 주의
- 법률 PDF 3개가 `../data/legal/` 에 있어야 합니다.
- 2단계가 `../db/legal_db/` 를 **새로 만듭니다.** 이미 있으면 건너뛰고(`REBUILD_DB = False`), 다시 만들려면 `True` 로 바꾸세요. 새로 만들 때는 임베딩 API 사용 요금이 발생합니다.
- 3단계의 Re-rank 모델(`BAAI/bge-reranker-v2-m3`)은 **처음 실행할 때 HuggingFace 에서 내려받습니다.** (용량이 크고 시간이 걸립니다. `sentence-transformers` 필요)
- 웹 검색에는 `TAVILY_API_KEY` 가 필요합니다.
- LLM·임베딩·웹 검색 호출은 사용 요금이 발생할 수 있습니다. 특히 4~6단계는 질문 하나에 LLM 을 수십 번 호출할 수 있습니다.

## 1. 환경 설정

`(1) API 키`

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 제공자별 API 키 (키 값은 출력하지 않음)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")  # 임베딩에 항상 필요

# Tavily 는 환경변수 TAVILY_API_KEY 를 라이브러리가 직접 읽음 — 설정 여부만 확인
print("TAVILY_API_KEY 설정됨:", bool(os.getenv("TAVILY_API_KEY")))

`(3) LLM 설정 (Groq / OpenAI / Upstage 선택)`
- `PROVIDER` 값만 바꾸면 이후 모든 셀의 모델이 바뀝니다. (Groq 는 OpenAI 호환 API 라서 `ChatOpenAI` 에 `base_url` 만 다르게 연결)
- 바꾼 뒤에는 이 셀부터 다시 실행하세요. OpenAI 선택 시 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "upstage"

# 제공자별 설정 (cls: 사용할 LangChain 채팅 모델 클래스)
LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o-mini",  # 테스트에는 작은 모델 사용 (대안: "gpt-4o")
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: 모델 클래스에 그대로 전달할 추가 옵션 (예: seed=42, timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    model_cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = model_cls(**config, temperature=temperature, max_retries=MAX_RETRIES, **kwargs)
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 구조화 출력·평가는 결과가 일정해야 하므로 temperature=0 사용
llm = get_llm(temperature=0)

`(2) 기본 라이브러리`

In [ ]:
import json
import re
import uuid
import warnings
from glob import glob
from operator import add
from pprint import pprint
from textwrap import dedent
from typing import Annotated, List, Literal, Optional, TypedDict

warnings.filterwarnings("ignore")

from langchain.agents import create_agent
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_upstage import UpstageEmbeddings
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

# 임베딩 모델: 벡터DB 를 만들 때와 검색할 때 반드시 같은 모델을 사용
EMBEDDING_MODEL = "solar-embedding-1-large"
embeddings_model = UpstageEmbeddings(model=EMBEDDING_MODEL)

In [ ]:
# 그래프 시각화 도우미: PNG 이미지를 시도하고, 실패하면 mermaid 코드를 출력
# (draw_mermaid_png 는 외부 서비스(mermaid.ink)를 호출하므로 네트워크가 막혀 있으면 실패할 수 있음)
from IPython.display import display, Image


def show_graph(compiled_graph) -> None:
    """컴파일된 그래프를 이미지로 보여 주고, 불가능하면 mermaid 코드를 출력합니다.
    mermaid 코드는 https://mermaid.live/ 에 붙여 넣으면 그림으로 볼 수 있습니다."""
    try:
        display(Image(compiled_graph.get_graph().draw_mermaid_png()))
    except Exception as e:
        print(f"이미지 생성 실패({type(e).__name__}) → mermaid 코드를 출력합니다.\n")
        print(compiled_graph.get_graph().draw_mermaid())

## 2. 법률 문서를 로드하여 벡터저장소에 저장
* 법령 PDF 는 `제1장 총칙 → 제1조(목적) ...` 구조입니다. 장·조 단위로 나누면 **조항 하나가 문서 하나**가 되어 검색 결과가 법 조문과 정확히 대응합니다.
* 세 법률은 처리 순서가 같고 **법 이름 · PDF 경로 · 저장 경로**만 다르므로, 설정(`LAW_CONFIGS`)과 공통 함수로 처리합니다.

In [ ]:
# 상수: 경로와 법률별 설정
LEGAL_DATA_DIR = "../data/legal"        # 법령 PDF 폴더
LEGAL_DB_DIR = "../db/legal_db"         # 벡터DB 저장 폴더
REBUILD_DB = False                      # True 이면 이미 있는 벡터DB 도 다시 만듦 (임베딩 요금 발생)

# 법률별 설정: PDF 파일명, 법 이름(헤더 제거·메타데이터에 사용), 저장 폴더, 검색 도구 이름
LAW_CONFIGS = {
    "personal": {
        "name": "개인정보 보호법",
        "pdf": "개인정보 보호법(법률)(제19234호)(20240315).pdf",
        "db": "personal_law",
        "tool": "personal_law_search",
    },
    "labor": {
        "name": "근로기준법",
        "pdf": "근로기준법(법률)(제18176호)(20211119).pdf",
        "db": "labor_law",
        "tool": "labor_law_search",
    },
    "housing": {
        "name": "주택임대차보호법",
        "pdf": "주택임대차보호법(법률)(제19356호)(20230719).pdf",
        "db": "housing_law",
        "tool": "housing_law_search",
    },
}

# PDF 파일 목록 확인
pdf_files = glob(os.path.join(LEGAL_DATA_DIR, "*.pdf"))
pprint(pdf_files)

`(1) 법령 텍스트를 장 · 조 단위로 분할`
* `parse_law()`: 정규식으로 **서문 / 장(章) / 조(條) / 부칙** 을 나눕니다.
    * 장이 있는 법(개인정보 보호법, 근로기준법) → `{"장": {장 제목: [조문, ...]}}`
    * 장이 없는 법(주택임대차보호법) → `{"조문": [조문, ...]}`
* PDF 의 매 페이지에 반복되는 머리글(`법제처 N 국가법령정보센터 법이름`)은 제거합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader


def parse_law(law_text: str) -> dict:
    """법령 전문 텍스트를 서문 · 장 · 조 · 부칙으로 분할합니다.

    장(제N장)이 있으면 결과에 '장' 키(장 제목 → 조문 리스트)를, 없으면 '조문' 키(조문 리스트)를 담습니다.
    """
    # 서문: 문서 처음부터 '제1장' 또는 '제1조' 직전까지
    preamble_pattern = r"^(.*?)(?=제1장|제1조)"
    preamble = re.search(preamble_pattern, law_text, re.DOTALL)
    preamble = preamble.group(1).strip() if preamble else None

    # 장: '제N장 제목' 과 그 뒤에 오는 모든 조항을 하나로 묶음
    chapter_pattern = r"(제\d+장\s+.+?)\n((?:제\d+조(?:의\d+)?(?:\(\w+\))?.*?)(?=제\d+장|부칙|$))"
    chapters = re.findall(chapter_pattern, law_text, re.DOTALL)

    # 부칙: '부칙' 으로 시작하는 나머지 전부
    appendix_pattern = r"(부칙.*)"
    appendix = re.search(appendix_pattern, law_text, re.DOTALL)
    appendix = appendix.group(1) if appendix else None

    parsed_law = {"서문": preamble, "부칙": appendix}

    # 조 분리: '제N조(제목)' 으로 시작해 다음 조가 나오기 전까지 (제N조의M 형식 포함)
    article_pattern = r"(제\d+조(?:의\d+)?\s*\([^)]+\).*?)(?=제\d+조(?:의\d+)?\s*\([^)]+\)|$)"

    if chapters:  # 장이 있는 경우
        parsed_law["장"] = {
            title.strip(): [a.strip() for a in re.findall(article_pattern, content, re.DOTALL)]
            for title, content in chapters
        }
    else:  # 장이 없는 경우: 서문과 부칙을 뺀 본문에서 조문 추출
        main_text = re.sub(preamble_pattern, "", law_text, flags=re.DOTALL)
        main_text = re.sub(appendix_pattern, "", main_text, flags=re.DOTALL)
        parsed_law["조문"] = [a.strip() for a in re.findall(article_pattern, main_text, re.DOTALL)]

    return parsed_law

`(2) 조항별 Document 생성`
* 조문 본문 앞에 **법 이름과 장 제목**을 붙여 둡니다. 임베딩에 이 정보가 반영되어 "근로기준법의 연차휴가" 같은 질문이 정확히 검색됩니다.
* 같은 정보를 `metadata`(`source`, `name`, `chapter`)에도 저장해 출처 표시에 사용합니다.

In [ ]:
def load_law_documents(law_key: str) -> List[Document]:
    """LAW_CONFIGS 의 법률 하나를 PDF 에서 읽어 조항 단위 Document 목록으로 만듭니다."""
    config = LAW_CONFIGS[law_key]
    pdf_file = os.path.join(LEGAL_DATA_DIR, config["pdf"])
    law_name = config["name"]

    pages = PyPDFLoader(pdf_file).load()

    # 각 페이지의 반복 머리글을 지우고 하나의 텍스트로 결합
    header_pattern = rf"법제처\s+\d+\s+국가법령정보센터\n{re.escape(law_name)}"
    law_text = "\n".join(re.sub(header_pattern, "", p.page_content).strip() for p in pages)

    parsed_law = parse_law(law_text)

    # (장 제목, 조문) 쌍으로 펼침: 장이 없으면 장 제목은 None
    if "장" in parsed_law:
        articles = [(chapter, a) for chapter, items in parsed_law["장"].items() for a in items]
    else:
        articles = [(None, a) for a in parsed_law["조문"]]

    docs = []
    for chapter, article in articles:
        metadata = {"source": pdf_file, "name": law_name}
        if chapter:
            metadata["chapter"] = chapter
            origin = f"{law_name} {chapter}"
        else:
            origin = law_name
        content = f"[법률정보]\n다음 조항은 {origin}에서 발췌한 내용입니다.\n\n[법률조항]\n{article}"
        docs.append(Document(page_content=content, metadata=metadata))
    return docs

In [ ]:
# 개인정보 보호법으로 분할 결과 확인 (임베딩 호출 없음)
personal_docs = load_law_documents("personal")

print("조항(Document) 수:", len(personal_docs))
print(personal_docs[0].page_content)
print(personal_docs[0].metadata)
print("-" * 60)
print(personal_docs[-1].page_content)
print(personal_docs[-1].metadata)

# 문서 길이의 최소·최대 확인 (너무 짧거나 긴 문서가 없는지 점검)
text_lengths = [len(d.page_content) for d in personal_docs]
print("문서 길이 최소/최대:", min(text_lengths), max(text_lengths))

`(3) 벡터저장소에 인덱싱`
* 세 법률을 FAISS 로 만들어 `../db/legal_db/` 에 저장합니다.
* 이미 저장된 인덱스는 **건너뜁니다**(`REBUILD_DB = False`). 다시 만들 때만 임베딩 요금이 발생합니다.

In [ ]:
os.makedirs(LEGAL_DB_DIR, exist_ok=True)

for law_key, config in LAW_CONFIGS.items():
    db_path = os.path.join(LEGAL_DB_DIR, config["db"])

    if os.path.exists(db_path) and not REBUILD_DB:
        print(f"[건너뜀] {config['name']}: 이미 있음 ({db_path})")
        continue

    docs = personal_docs if law_key == "personal" else load_law_documents(law_key)
    law_db = FAISS.from_documents(documents=docs, embedding=embeddings_model)  # 임베딩 API 호출
    law_db.save_local(db_path)
    print(f"[저장] {config['name']}: {len(docs)}개 조항 → {db_path}")

## 3. 도구 호출
### 3-1. 법률 정보 검색 도구, 웹 검색 도구 정의

* `ContextualCompressionRetriever`: 기본 검색 결과를 **압축 · 필터링**해 더 관련성 높은 문서만 돌려주는 고급 Retriever 입니다.
* `CrossEncoderReranker`: Cross-Encoder 모델로 검색된 문서를 **재순위(re-ranking)** 하는 문서 압축기입니다. 질문과 문서를 한 쌍으로 넣어 점수를 매기므로 벡터 유사도만 쓸 때보다 정확합니다.
* `HuggingFaceCrossEncoder`: HuggingFace 의 Cross-Encoder 모델을 LangChain 에서 쓰도록 감싼 클래스입니다.
* 검색은 **벡터로 5건 가져온 뒤 Re-rank 로 상위 2건**만 남기는 2단계입니다.
* 세 법률 도구는 구조가 같으므로 도구를 만들어 주는 함수(`make_law_search_tool`)로 한 번에 만듭니다.

In [ ]:
# langchain 1.x 에서 retrievers 는 langchain_classic 으로 이동했음
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.retrievers import TavilySearchAPIRetriever

# 상수
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"  # Re-rank(Cross-Encoder) 모델
RERANK_TOP_N = 2        # Re-rank 후 남길 문서 수
LAW_SEARCH_K = 5        # 법률 벡터 검색에서 가져올 문서 수
WEB_SEARCH_K = 10       # 웹 검색에서 가져올 문서 수

# Re-rank 모델 (처음 실행 시 모델을 내려받음)
rerank_model = HuggingFaceCrossEncoder(model_name=RERANK_MODEL_NAME)
cross_reranker = CrossEncoderReranker(model=rerank_model, top_n=RERANK_TOP_N)


def make_law_search_tool(tool_name: str, law_name: str, db_path: str):
    """법률 하나의 벡터DB 를 로드해 '검색 → Re-rank' 도구를 만들어 반환합니다.

    tool_name: LLM 에게 보이는 도구 이름 / law_name: 도구 설명에 쓸 법 이름 / db_path: FAISS 폴더
    """
    law_db = FAISS.load_local(db_path, embeddings_model, allow_dangerous_deserialization=True)
    retriever = ContextualCompressionRetriever(
        base_compressor=cross_reranker,
        base_retriever=law_db.as_retriever(search_kwargs={"k": LAW_SEARCH_K}),
    )

    @tool(tool_name, description=f"{law_name} 법률 조항을 검색합니다.")
    def law_search(query: str) -> List[Document]:
        docs = retriever.invoke(query)
        return docs or [Document(page_content="관련 정보를 찾을 수 없습니다.")]

    return law_search


# 법률별 검색 도구 생성
personal_law_search = make_law_search_tool(
    LAW_CONFIGS["personal"]["tool"], LAW_CONFIGS["personal"]["name"],
    os.path.join(LEGAL_DB_DIR, LAW_CONFIGS["personal"]["db"]))
labor_law_search = make_law_search_tool(
    LAW_CONFIGS["labor"]["tool"], LAW_CONFIGS["labor"]["name"],
    os.path.join(LEGAL_DB_DIR, LAW_CONFIGS["labor"]["db"]))
housing_law_search = make_law_search_tool(
    LAW_CONFIGS["housing"]["tool"], LAW_CONFIGS["housing"]["name"],
    os.path.join(LEGAL_DB_DIR, LAW_CONFIGS["housing"]["db"]))

# 웹 검색: Tavily 로 10건을 가져와 Re-rank 로 상위 2건만 사용
web_retriever = ContextualCompressionRetriever(
    base_compressor=cross_reranker,
    base_retriever=TavilySearchAPIRetriever(k=WEB_SEARCH_K),
)


@tool
def web_search(query: str) -> List[Document]:
    """데이터베이스에 없는 정보 또는 최신 정보를 웹에서 검색합니다."""
    docs = web_retriever.invoke(query)

    formatted_docs = [
        Document(
            page_content=f'<Document href="{doc.metadata["source"]}"/>\n{doc.page_content}\n</Document>',
            metadata={"source": "web search", "url": doc.metadata["source"]},
        )
        for doc in docs
    ]
    return formatted_docs or [Document(page_content="관련 정보를 찾을 수 없습니다.")]


# 도구 목록
tools = [personal_law_search, labor_law_search, housing_law_search, web_search]
print([t.name for t in tools])

### 3-2. LLM 의 도구 선택 확인
* `bind_tools` 로 도구를 연결하면 LLM 이 질문에 맞는 도구를 **스스로** 고릅니다. 이 셀은 도구를 실행하지 않고 선택(`tool_calls`)만 확인합니다.
* 4-2 에서는 같은 일을 **구조화 출력 라우터**로 명시적으로 처리합니다.

In [ ]:
llm_with_tools = llm.bind_tools(tools)  # 도구 설명이 포함된 LLM

test_questions = [
    "연차휴가 부여 기준에 대해서 설명해주세요.",                                    # 근로기준법 검색
    "안녕하세요?",                                                              # 도구 없음
    "연차휴가 부여 기준에 대해서 설명해주세요. 2023년 연차휴가 사용 비율은 어느 정도인가요?",  # 법률 + 웹 검색
    "전월세 직거래 시에 유의사항은 무엇인가요?",                                    # 주택임대차 + 웹 검색
]

for question in test_questions:
    ai_msg = llm_with_tools.invoke(question)
    print(f"질문: {question}")
    print("  tool_calls:", [(tc["name"], tc["args"]) for tc in ai_msg.tool_calls] or "없음")

## 4. Agent RAG 구현

### 4-1. 각 법률에 특화된 RAG 에이전트 (Corrective RAG)
검색된 문서의 **관련성을 평가**하고, 부족하면 **질문을 고쳐 다시 검색**하는 에이전트입니다.

| 노드 | 하는 일 |
|---|---|
| `retrieve` | 질문(또는 재작성된 질문)으로 검색 도구 호출 |
| `extract_and_evaluate` | 문서마다 LLM 이 핵심 정보를 뽑고 **관련성 · 충실성 점수**(0~1)를 매김 → 기준 이상만 남김 |
| `rewrite_query` | 정보가 부족하면 검색 질문을 개선 |
| `generate_answer` | 남은 정보를 근거로 출처가 포함된 답변 생성 |

* 라우팅(`should_continue`): 기준을 통과한 정보가 **1개 이상이면 종료**, 없으면 질문을 고쳐 재시도합니다. 재시도는 **최대 2회**(`MAX_GENERATIONS`)까지입니다.
* 법률 3종과 웹 검색, 총 4개의 에이전트가 **프롬프트의 전문 분야 표현만 다르고 구조는 같으므로** 에이전트를 만들어 주는 함수(`build_corrective_rag_agent`) 하나로 만듭니다.

In [ ]:
# 상수: 정보 평가 기준
MIN_QUERY_RELEVANCE = 0.8   # 문서 전체의 질문 답변 가능성이 이 값 미만이면 문서를 버림
MIN_STRIP_SCORE = 0.7       # 정보 조각의 관련성·충실성이 이 값 이하이면 버림
MAX_GENERATIONS = 2         # 정보 추출 최대 횟수 (무한 루프 방지)


class InformationStrip(BaseModel):
    """추출된 정보에 대한 내용과 출처, 관련성 점수"""
    content: str = Field(..., description="추출된 정보 내용")
    source: str = Field(..., description="정보의 출처(법률 조항 또는 URL 등). 예시: 환경법 제22조 3항 or 블로그 환경법 개정 (https://blog.com/page/123)")
    relevance_score: float = Field(..., ge=0, le=1, description="관련성 점수 (0에서 1 사이)")
    faithfulness_score: float = Field(..., ge=0, le=1, description="충실성 점수 (0에서 1 사이)")


class ExtractedInformation(BaseModel):
    """문서 하나에서 추출한 정보 조각들과 전반적인 답변 가능성"""
    strips: List[InformationStrip] = Field(..., description="추출된 정보 조각들")
    query_relevance: float = Field(..., ge=0, le=1, description="질의에 대한 전반적인 답변 가능성 점수 (0에서 1 사이)")


class RefinedQuestion(BaseModel):
    """개선된 질문과 이유"""
    question_refined: str = Field(..., description="개선된 질문")
    reason: str = Field(..., description="이유")


class CorrectiveRagState(TypedDict):
    """Corrective RAG 에이전트의 상태 (4개 에이전트가 공통으로 사용)"""
    question: str                           # 사용자의 질문
    documents: List[Document]               # 검색된 문서
    num_generations: int                    # 정보 추출 횟수 (무한 루프 방지에 활용)
    rewritten_query: str                    # 재작성한 질문
    extracted_info: List[InformationStrip]  # 기준을 통과한 정보 조각
    node_answer: Optional[str]              # 이 에이전트의 최종 답변

In [ ]:
# 프롬프트 템플릿: @EXPERT@ 등 자리표시자는 에이전트를 만들 때 채워짐 ({question} 등은 호출 시 채워지는 변수)
EXTRACT_SYSTEM = """당신은 @EXPERT@입니다. 주어진 문서에서 질문과 관련된 주요 사실과 정보를 3~5개 정도 추출하세요.
각 추출된 정보에 대해 다음 두 가지 측면을 0에서 1 사이의 점수로 평가하세요:
1. 질문과의 관련성
2. 답변의 충실성 (질문에 대한 완전하고 정확한 답변을 제공할 수 있는 정도)

마지막으로, 추출된 정보를 종합하여 질문에 대한 전반적인 답변 가능성을 0에서 1 사이의 점수로 평가하세요."""

REWRITE_SYSTEM = """당신은 @EXPERT@입니다. 주어진 원래 질문과 추출된 정보를 바탕으로, 더 관련성 있고 충실한 정보를 찾기 위해 검색 쿼리를 개선해주세요.

다음 사항을 고려하여 검색 쿼리를 개선하세요:
1. 원래 질문의 핵심 요소
2. 추출된 정보의 관련성 점수
3. 추출된 정보의 충실성 점수
4. 부족한 정보나 더 자세히 알아야 할 부분

개선된 검색 쿼리 작성 단계:
1. 2-3개의 검색 쿼리를 제안하세요.
2. 각 쿼리는 구체적이고 간결해야 합니다(5-10 단어 사이).
3. @TERMS@을 적절히 활용하세요.
4. 각 쿼리 뒤에는 해당 쿼리를 제안한 이유를 간단히 설명하세요.

마지막으로, 제안된 쿼리 중 가장 효과적일 것 같은 쿼리 하나를 선택하세요."""

ANSWER_SYSTEM = """당신은 @EXPERT@입니다. 주어진 질문과 추출된 정보를 바탕으로 답변을 생성해주세요.
답변은 마크다운 형식으로 작성하며, 각 정보의 출처를 명확히 표시해야 합니다.
답변 구조:
1. 질문에 대한 직접적인 답변
2. @SECTION2@
3. 추가 설명 또는 예시 (필요한 경우)
4. 결론 및 요약
각 섹션에서 사용된 정보의 출처를 괄호 안에 명시하세요. 예: (출처: @CITE@)"""

NO_INFO_MESSAGE = "관련 정보를 찾을 수 없어 답변을 생성하지 못했습니다."  # 기준을 통과한 정보가 없을 때


def _fill(template: str, **values: str) -> str:
    """템플릿의 @KEY@ 자리표시자를 값으로 바꿉니다."""
    for key, value in values.items():
        template = template.replace(f"@{key}@", value)
    return template

In [ ]:
def build_corrective_rag_agent(search_tool, expert: str, terms: str, section2: str, cite_example: str):
    """검색 도구 하나로 Corrective RAG 에이전트(컴파일된 그래프)를 만들어 반환합니다.

    search_tool: 검색 도구 / expert: 전문가 역할(예: '근로기준법 전문가') / terms: 쿼리 재작성 시 활용할 용어 설명
    section2: 답변 2번 항목 제목 / cite_example: 출처 표기 예시
    """
    values = dict(EXPERT=expert, TERMS=terms, SECTION2=section2, CITE=cite_example)

    extract_chain = (
        ChatPromptTemplate.from_messages([
            ("system", _fill(EXTRACT_SYSTEM, **values)),
            ("human", "[질문]\n{question}\n\n[문서 내용]\n{document_content}"),
        ])
        | llm.with_structured_output(ExtractedInformation)
    )
    rewrite_chain = (
        ChatPromptTemplate.from_messages([
            ("system", _fill(REWRITE_SYSTEM, **values)),
            ("human", "원래 질문: {question}\n\n추출된 정보:\n{extracted_info}\n\n위 지침에 따라 개선된 검색 쿼리를 작성해주세요."),
        ])
        | llm.with_structured_output(RefinedQuestion)
    )
    answer_chain = (
        ChatPromptTemplate.from_messages([
            ("system", _fill(ANSWER_SYSTEM, **values)),
            ("human", "질문: {question}\n\n추출된 정보:\n{extracted_info}\n\n위 지침에 따라 최종 답변을 작성해주세요."),
        ])
        | llm
        | StrOutputParser()
    )

    def retrieve_documents(state: CorrectiveRagState) -> dict:
        """재작성된 질문이 있으면 그것으로, 없으면 원래 질문으로 검색합니다."""
        print("---문서 검색---")
        query = state.get("rewritten_query") or state["question"]
        return {"documents": search_tool.invoke(query)}

    def extract_and_evaluate_information(state: CorrectiveRagState) -> dict:
        """문서마다 정보를 추출·평가하고, 기준을 통과한 정보 조각만 남깁니다."""
        print("---정보 추출 및 평가---")
        extracted_strips = []

        for doc in state["documents"]:
            extracted = extract_chain.invoke({
                "question": state["question"],
                "document_content": doc.page_content,
            })
            if extracted.query_relevance < MIN_QUERY_RELEVANCE:
                continue  # 이 문서로는 질문에 답하기 어려움
            extracted_strips.extend(
                strip for strip in extracted.strips
                if strip.relevance_score > MIN_STRIP_SCORE and strip.faithfulness_score > MIN_STRIP_SCORE
            )

        return {
            "extracted_info": extracted_strips,
            "num_generations": state.get("num_generations", 0) + 1,
        }

    def rewrite_query(state: CorrectiveRagState) -> dict:
        """정보가 부족할 때 검색 질문을 개선합니다."""
        print("---쿼리 재작성---")
        extracted_info_str = "\n".join(strip.content for strip in state["extracted_info"])
        response = rewrite_chain.invoke({"question": state["question"], "extracted_info": extracted_info_str})
        return {"rewritten_query": response.question_refined}

    def generate_node_answer(state: CorrectiveRagState) -> dict:
        """기준을 통과한 정보로 답변을 생성합니다. 정보가 없으면 지어내지 않고 안내 문구를 반환합니다."""
        print("---답변 생성---")
        if not state["extracted_info"]:
            return {"node_answer": NO_INFO_MESSAGE}

        extracted_info_str = "\n".join(
            f"내용: {s.content}\n출처: {s.source}\n관련성: {s.relevance_score}\n충실성: {s.faithfulness_score}"
            for s in state["extracted_info"]
        )
        return {"node_answer": answer_chain.invoke({"question": state["question"], "extracted_info": extracted_info_str})}

    def should_continue(state: CorrectiveRagState) -> Literal["계속", "종료"]:
        """정보가 충분하거나 재시도 한도에 도달하면 '종료'(→ 답변 생성), 아니면 '계속'(→ 질문 재작성)."""
        if state["num_generations"] >= MAX_GENERATIONS or len(state["extracted_info"]) >= 1:
            return "종료"
        return "계속"

    workflow = StateGraph(CorrectiveRagState)
    workflow.add_node("retrieve", retrieve_documents)
    workflow.add_node("extract_and_evaluate", extract_and_evaluate_information)
    workflow.add_node("rewrite_query", rewrite_query)
    workflow.add_node("generate_answer", generate_node_answer)

    workflow.add_edge(START, "retrieve")
    workflow.add_edge("retrieve", "extract_and_evaluate")
    workflow.add_conditional_edges(
        "extract_and_evaluate",
        should_continue,
        {"계속": "rewrite_query", "종료": "generate_answer"},
    )
    workflow.add_edge("rewrite_query", "retrieve")  # 재작성한 질문으로 다시 검색 (루프)
    workflow.add_edge("generate_answer", END)
    return workflow.compile()

`(1) 법률별 에이전트와 웹 검색 에이전트 생성`
* 아래 4개는 모두 위 함수로 만든 같은 구조의 그래프입니다. 전문 분야와 출처 표기 예시만 다릅니다.

In [ ]:
# (1) 개인정보보호법 에이전트
personal_law_agent = build_corrective_rag_agent(
    personal_law_search, expert="개인정보보호법 전문가", terms="개인정보보호법과 관련된 전문 용어",
    section2="관련 법률 조항 및 해석", cite_example="개인정보 보호법 제15조")

# (2) 근로기준법 에이전트
labor_law_agent = build_corrective_rag_agent(
    labor_law_search, expert="근로기준법 전문가", terms="근로기준법과 관련된 전문 용어",
    section2="관련 법률 조항 및 해석", cite_example="근로기준법 제15조")

# (3) 주택임대차보호법 에이전트
housing_law_agent = build_corrective_rag_agent(
    housing_law_search, expert="주택임대차보호법 전문가", terms="주택임대차보호법과 관련된 전문 용어",
    section2="관련 법률 조항 및 해석", cite_example="주택임대차보호법 제3조")

# (4) 웹 검색 에이전트
search_web_agent = build_corrective_rag_agent(
    web_search, expert="인터넷 정보 검색 전문가", terms="질문과 관련된 전문 용어",
    section2="관련 출처 및 링크", cite_example="블로그 (www.blog.com/page/001)")

In [ ]:
show_graph(personal_law_agent)  # 4개 에이전트의 구조는 모두 같음

* 점선(`-.->`)이 조건부 엣지(`계속` / `종료`)입니다. `rewrite_query → retrieve` 실선이 재검색 루프를 만듭니다.

In [ ]:
def run_stream(graph, inputs, config=None) -> dict:
    """그래프를 stream 으로 실행하며 노드별 결과를 요약 출력하고, 누적된 최종 상태(dict)를 반환합니다.
    입력값(question 등)도 결과에 포함됩니다. inputs=None 이면 중단된 실행을 이어서 진행합니다."""
    final_state = dict(inputs) if isinstance(inputs, dict) else {}
    for event in graph.stream(inputs, config=config, stream_mode="updates"):
        for node, update in event.items():
            if node.startswith("__") or not isinstance(update, dict):  # __interrupt__ 등 내부 이벤트 제외
                continue
            final_state.update(update)
            summary = {k: (f"{len(v)}건" if isinstance(v, list) else v) for k, v in update.items()
                       if k in ("datasources", "documents", "extracted_info", "rewritten_query", "num_generations")}
            print(f"[{node}] {summary}")
    print("-" * 60)
    return final_state


result = run_stream(personal_law_agent, {"question": "개인정보 처리에 대한 동의를 받을 때 주의해야 할 점은 무엇인가요?"})
print(result["node_answer"])

In [ ]:
result = run_stream(labor_law_agent, {"question": "근로계약 체결할 때 주의해야 할 점은 무엇인가요?"})
print(result["node_answer"])

In [ ]:
result = run_stream(housing_law_agent, {"question": "대리인과 아파트 임대차 계약을 체결할 때 주의해야 할 점은 무엇인가요?"})
print(result["node_answer"])

In [ ]:
result = run_stream(search_web_agent, {"question": "대리인과 아파트 임대차 계약을 체결할 때 주의해야 할 점은 무엇인가요?"})
print(result["node_answer"])

### 4-2. 질문 라우팅
* 사용자의 질문을 분석해 적절한 에이전트를 **1개 이상** 선택합니다(Adaptive RAG). 선택된 에이전트는 **병렬**로 실행되고, 결과는 `answers` 에 모입니다.
* `answers` 는 병렬 노드의 결과를 모으려고 **커스텀 reducer**(`merge_answers`)를 씁니다. 분석 단계가 시작될 때마다 목록을 **초기화**해서, 같은 대화에서 여러 번 질문하거나 거절 후 다시 시도해도 이전 답변이 섞이지 않습니다.
* 검색이 필요 없는 질문(인사 등)은 `llm_fallback` 으로 보내 LLM 이 바로 답합니다.

In [ ]:
RESET_ANSWERS = "__reset_answers__"  # answers 를 초기화하라는 표시 값


def merge_answers(old: Optional[List[str]], new: Optional[List[str]]) -> List[str]:
    """answers 의 reducer: 보통은 이어 붙이고, 첫 값이 RESET_ANSWERS 이면 나머지로 교체(초기화)합니다."""
    old = old or []
    new = new or []
    if new and new[0] == RESET_ANSWERS:
        return new[1:]
    return old + new


class ResearchAgentState(TypedDict):
    """메인 그래프의 상태"""
    question: str
    answers: Annotated[List[str], merge_answers]  # 각 전문가 에이전트의 답변 (병렬 결과가 모임)
    final_answer: str
    datasources: List[str]                        # 라우터가 선택한 경로들
    evaluation_report: Optional[dict]             # 답변 평가 결과 (5단계)
    user_decision: Optional[str]                  # 사람의 승인 결과 'approved' / 'rejected' (5~6단계)

In [ ]:
class ToolSelector(BaseModel):
    """Routes the user question to the most appropriate tool."""
    tool: Literal["search_personal", "search_labor", "search_housing", "search_web", "llm_fallback"] = Field(
        description="Select one of the tools, based on the user's question.",
    )


class ToolSelectors(BaseModel):
    """Select the appropriate tools that are suitable for the user question."""
    tools: List[ToolSelector] = Field(
        description="Select one or more tools, based on the user's question.",
    )


# 구조화 출력 LLM: 응답이 ToolSelectors 객체로 반환됨
structured_llm_tool_selector = llm.with_structured_output(ToolSelectors)

ROUTER_SYSTEM_PROMPT = dedent("""You are an AI assistant specializing in routing user questions to the appropriate tools.
Use the following guidelines:
- For questions specifically about legal provisions or articles of the privacy protection law (개인정보 보호법), use the search_personal tool.
- For questions specifically about legal provisions or articles of the labor law (근로기준법), use the search_labor tool.
- For questions specifically about legal provisions or articles of the housing law (주택임대차보호법), use the search_housing tool.
- For any other information, including questions related to these laws but not directly about specific legal provisions, or for the most up-to-date data, use the search_web tool.
- For greetings or small talk that need no search, use only the llm_fallback tool.
Always choose all of the appropriate tools based on the user's question.
If a question is about a law but doesn't seem to be asking about specific legal provisions, include both the relevant law search tool and the search_web tool.""")

route_prompt = ChatPromptTemplate.from_messages([
    ("system", ROUTER_SYSTEM_PROMPT),
    ("human", "{question}"),
])

# 질문 라우터: 프롬프트 → 구조화 출력 LLM
question_tool_router = route_prompt | structured_llm_tool_selector

# 라우팅 결과 확인
router_tests = [
    "근로계약 체결할 때 개인정보 취급 상의 유의사항은 무엇인가요?",   # search_personal + search_labor
    "법에서 정한 연차휴가 기준을 알려주세요.",                        # search_labor (+ search_web)
    "개인정보보호법에서 정한 가명정보의 정의는 무엇인가요?",          # search_personal
    "안녕하세요?",                                                 # llm_fallback
]
for question in router_tests:
    print(f"{question} → {[t.tool for t in question_tool_router.invoke({'question': question}).tools]}")

In [ ]:
VALID_SOURCES = ("search_personal", "search_labor", "search_housing", "search_web", "llm_fallback")


def analyze_question_tool_search(state: ResearchAgentState) -> dict:
    """질문을 분석해 사용할 경로(datasources)를 정하고, 이전 답변·승인 결과를 초기화합니다."""
    try:
        result = question_tool_router.invoke({"question": state["question"]})
        datasources = [t.tool for t in result.tools]
    except Exception as e:  # 라우팅 실패 시 안전하게 LLM 직접 답변으로
        print(f"Error in routing: {e}")
        datasources = ["llm_fallback"]
    # answers 초기화(RESET_ANSWERS): 재시도·후속 질문에서 이전 답변이 섞이지 않게 함
    return {"datasources": datasources, "answers": [RESET_ANSWERS], "user_decision": None}


def route_datasources_tool_search(state: ResearchAgentState) -> List[str]:
    """라우팅 함수: 선택된 경로 이름 목록을 반환합니다. 여러 개면 해당 노드가 병렬 실행됩니다.

    - 유효하지 않은 값은 버리고 중복은 제거(순서 유지)
    - 검색 경로와 llm_fallback 이 함께 선택되면 검색 경로만 사용
    - 아무것도 남지 않으면 llm_fallback
    """
    selected = list(dict.fromkeys(s for s in state["datasources"] if s in VALID_SOURCES))
    if len(selected) > 1:
        selected = [s for s in selected if s != "llm_fallback"]
    return selected or ["llm_fallback"]

In [ ]:
def make_expert_node(agent, label: str):
    """전문가 에이전트(서브그래프)를 메인 그래프의 노드로 감싸는 함수를 만듭니다.
    에이전트의 답변(node_answer)을 answers 에 추가합니다."""

    def expert_node(state: ResearchAgentState) -> dict:
        print(f"--- {label} 에이전트 시작 ---")
        answer = agent.invoke({"question": state["question"]})
        return {"answers": [answer["node_answer"]]}

    return expert_node


personal_rag_node = make_expert_node(personal_law_agent, "개인정보보호법 전문가")
labor_rag_node = make_expert_node(labor_law_agent, "근로기준법 전문가")
housing_rag_node = make_expert_node(housing_law_agent, "주택임대차보호법 전문가")
web_rag_node = make_expert_node(search_web_agent, "인터넷 검색 전문가")

In [ ]:
# 최종 답변 생성 노드: 전문가들의 답변을 모아 하나의 답변으로 정리
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an assistant answering questions based on provided documents. Follow these guidelines:

1. Use only information from the given documents.
2. If the document lacks relevant info, say "제공된 정보로는 충분한 답변을 할 수 없습니다."
3. Cite the source of information for each sentence in your answer. Use the following format:
    - For legal articles: "법률명 제X조 Y항"
    - For web sources: "출처 제목 (URL)"
4. Don't speculate or add information not in the documents.
5. Keep answers concise and clear.
6. Omit irrelevant information.
7. If multiple sources provide the same information, cite all relevant sources.
8. If information comes from multiple sources, combine them coherently while citing each source.

Example of citation usage:
"부동산 거래 시 계약서에 거래 금액을 명시해야 합니다 (부동산 거래신고 등에 관한 법률 제3조 1항). 또한, 계약 체결일로부터 30일 이내에 신고해야 합니다 (부동산 거래 신고 안내 블로그, https://example.com/realestate)."
"""),
    ("human", "Answer the following question using these documents:\n\n[Documents]\n{documents}\n\n[Question]\n{question}"),
])

# LLM Fallback 프롬프트: 검색 없이 LLM 의 일반 지식으로 답변
fallback_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant helping with various topics. Follow these guidelines:

1. Provide accurate and helpful information to the best of your ability.
2. Express uncertainty when unsure; avoid speculation.
3. Keep answers concise yet informative.
4. Respond ethically and constructively.
5. Mention reliable general sources when applicable."""),
    ("human", "{question}"),
])


def answer_final(state: ResearchAgentState) -> dict:
    """전문가 답변(answers)을 근거로 최종 답변(final_answer)을 생성합니다."""
    print("---최종 답변---")
    documents_text = "\n\n".join(state.get("answers", []))
    rag_chain = rag_prompt | llm | StrOutputParser()
    generation = rag_chain.invoke({"documents": documents_text, "question": state["question"]})
    return {"final_answer": generation}


def llm_fallback(state: ResearchAgentState) -> dict:
    """검색 없이 LLM 이 직접 답변합니다."""
    print("---Fallback 답변---")
    llm_chain = fallback_prompt | llm | StrOutputParser()
    return {"final_answer": llm_chain.invoke({"question": state["question"]})}

`메인 그래프 구성`
* 같은 그래프를 5단계(평가 + 콘솔 승인), 6단계(중단점 + Gradio 승인)에서도 조금씩 바꿔 씁니다. 그래서 그래프 구성을 `build_legal_rag_graph()` 함수로 만들고 `review` 옵션으로 단계를 고릅니다. (이 단계는 `review=None`: 평가 없음)
* 검색 노드는 각각 `generate_answer` 로 연결됩니다. 병렬로 실행된 노드들이 **같은 단계에서 모두 끝난 뒤** `generate_answer` 가 한 번 실행됩니다.

In [ ]:
def build_legal_rag_graph(review: Optional[str] = None, checkpointer=None):
    """생활법률 RAG 메인 그래프를 구성해 컴파일합니다.

    review=None        : 답변 생성 후 바로 종료 (4-2)
    review="console"   : 답변 평가 → 콘솔 입력(input)으로 승인/거절 (5-2)
    review="breakpoint": 답변 평가 → 'human_review' 직전에 중단(interrupt_before), 외부(Gradio)에서 승인 (6)
    """
    builder = StateGraph(ResearchAgentState)

    # 공통 노드
    builder.add_node("analyze_question", analyze_question_tool_search)
    builder.add_node("search_personal", personal_rag_node)
    builder.add_node("search_labor", labor_rag_node)
    builder.add_node("search_housing", housing_rag_node)
    builder.add_node("search_web", web_rag_node)
    builder.add_node("generate_answer", answer_final)
    builder.add_node("llm_fallback", llm_fallback)

    # 공통 엣지: 질문 분석 → (선택된 경로들을 병렬 실행) → 답변 생성
    builder.add_edge(START, "analyze_question")
    builder.add_conditional_edges(
        "analyze_question",
        route_datasources_tool_search,
        ["search_personal", "search_labor", "search_housing", "search_web", "llm_fallback"],
    )
    for node in ["search_personal", "search_labor", "search_housing", "search_web"]:
        builder.add_edge(node, "generate_answer")
    builder.add_edge("llm_fallback", END)

    # 답변 이후 단계: review 옵션에 따라 달라짐
    if review is None:
        builder.add_edge("generate_answer", END)
    else:
        builder.add_node("evaluate_answer", evaluate_answer_node)
        builder.add_edge("generate_answer", "evaluate_answer")

        if review == "console":
            # 평가 결과를 보여 주고 콘솔에서 승인 여부를 입력받아 분기
            builder.add_conditional_edges(
                "evaluate_answer", human_review_console, {"approved": END, "rejected": "analyze_question"})
        elif review == "breakpoint":
            # human_review 노드 직전에 중단 → 외부에서 user_decision 을 채운 뒤 재개
            builder.add_node("human_review", lambda state: {})
            builder.add_edge("evaluate_answer", "human_review")
            builder.add_conditional_edges(
                "human_review",
                lambda state: "approved" if state.get("user_decision") == "approved" else "rejected",
                {"approved": END, "rejected": "analyze_question"},
            )
        else:
            raise ValueError(f"지원하지 않는 review 옵션: {review}")

    interrupt_before = ["human_review"] if review == "breakpoint" else None
    return builder.compile(checkpointer=checkpointer, interrupt_before=interrupt_before)


rag_search_graph = build_legal_rag_graph()
show_graph(rag_search_graph)

* 점선이 조건부 엣지입니다. `analyze_question` 에서 선택된 노드들만 병렬로 실행됩니다.

In [ ]:
question = "대리인과 아파트 임대차 계약을 체결할 때 주의해야 할 점은 무엇인가요?"
result = run_stream(rag_search_graph, {"question": question})
print(result["final_answer"])

## 5. 답변 평가 및 확인
### 5-1. 답변을 평가하는 ReAct 에이전트
* 평가 에이전트는 질문과 답변을 읽고 6개 기준(각 10점, **60점 만점**)으로 점수를 매깁니다. 필요하면 검색 도구로 직접 사실을 확인합니다.
* 평가 지침은 에이전트를 만들 때 `system_prompt` 로 지정합니다. 결과는 JSON 으로만 출력하게 합니다.
* 이 에이전트는 `create_agent`(ReAct 방식)로 만듭니다. (04번 노트북 참고)

In [ ]:
EVALUATION_PROMPT = dedent("""
당신은 AI 어시스턴트가 생성한 답변을 평가하는 전문가입니다. 주어진 질문과 답변을 평가하고, 60점 만점으로 점수를 매기세요. 다음 기준을 사용하여 평가하십시오:

1. 정확성 (10점)
2. 관련성 (10점)
3. 완전성 (10점)
4. 인용 정확성 (10점)
5. 명확성과 간결성 (10점)
6. 객관성 (10점)

평가 과정:
1. 주어진 질문과 답변을 주의 깊게 읽으십시오.
2. 필요한 경우, 제공된 검색 도구(웹 검색, 개인정보보호법·근로기준법·주택임대차보호법 검색)로 추가 정보를 확인하세요.
3. 각 기준에 대해 1-10점 사이의 점수를 매기세요.
4. 총점을 계산하세요 (60점 만점).

**중요: 최종 출력은 반드시 아래 JSON 형식으로만 작성하세요. 다른 텍스트는 포함하지 마세요.**

```json
{
  "scores": {
    "accuracy": 0,
    "relevance": 0,
    "completeness": 0,
    "citation_accuracy": 0,
    "clarity_conciseness": 0,
    "objectivity": 0
  },
  "total_score": 0,
  "brief_evaluation": "간단한 평가 설명"
}
```
""").strip()

# 평가 에이전트: 검색 도구를 가진 ReAct 에이전트
answer_reviewer = create_agent(model=llm, tools=tools, system_prompt=EVALUATION_PROMPT)

In [ ]:
show_graph(answer_reviewer)

In [ ]:
def safe_json_parse(text: str) -> dict:
    """AI 응답에서 JSON 을 안전하게 추출합니다. 파싱에 실패하면 총점 0 의 기본 구조를 반환합니다."""
    candidates = [text]
    # ```json ... ``` 코드 블록, 또는 가장 바깥 { ... } 구간도 후보로 시도
    for pattern in (r"```json\s*(.*?)\s*```", r"\{.*\}"):
        m = re.search(pattern, text, re.DOTALL)
        if m:
            candidates.append(m.group(1) if m.lastindex else m.group(0))

    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            continue

    return {"total_score": 0, "brief_evaluation": f"JSON 파싱 실패. 원본 응답: {text[:200]}..."}


def review_answer(question: str, answer: str) -> dict:
    """질문과 답변을 평가 에이전트에 전달하고, 평가 결과(dict)를 반환합니다."""
    message = HumanMessage(content=f"[질문]\n{question}\n\n[답변]\n{answer}")
    response = answer_reviewer.invoke({"messages": [message]})
    return safe_json_parse(response["messages"][-1].content)


# 직전 4-2 실행 결과(result)를 평가
report = review_answer(result["question"], result["final_answer"])
pprint(report)

### 5-2. 답변 평가 내용을 확인하는 HITL 추가
* `evaluate_answer` 노드가 답변을 평가해 `evaluation_report` 에 저장합니다.
* 사람이 평가 결과를 보고 승인(`y`)하면 종료, 거절(`n`)하면 **질문 분석 단계부터 다시** 실행합니다.
* 이 예는 노드 안에서 `input()` 으로 입력을 받는 가장 단순한 방식입니다. (6단계에서 웹 UI 용으로 바꿉니다)
* TODO: 거절 후 재시도는 같은 질문을 다시 실행하므로 결과가 달라지지 않을 수 있습니다. 실무에서는 사람의 거절 사유를 상태에 담아 다음 시도에 반영해야 합니다.

In [ ]:
def evaluate_answer_node(state: ResearchAgentState) -> dict:
    """생성된 최종 답변을 평가 에이전트로 평가해 evaluation_report 에 저장합니다."""
    print("---답변 평가---")
    return {"evaluation_report": review_answer(state["question"], state["final_answer"])}


def human_review_console(state: ResearchAgentState) -> Literal["approved", "rejected"]:
    """라우팅 함수: 답변과 평가 결과를 보여 주고, 콘솔 입력(y/n)으로 승인 여부를 결정합니다."""
    report = state["evaluation_report"]
    print("\n현재 답변:")
    print(state["final_answer"])
    print("\n평가 결과:")
    print(f"총점: {report.get('total_score', 0)}/60")
    print(f"간단 평가: {report.get('brief_evaluation', '정보 없음')}")

    user_input = input("\n이 답변을 승인하시겠습니까? (y/n): ").strip().lower()
    return "approved" if user_input == "y" else "rejected"


legal_rag_agent = build_legal_rag_graph(review="console")
show_graph(legal_rag_agent)

* 실행하면 평가 후 입력창이 나타납니다. `y` 를 입력하면 종료되고, `n` 을 입력하면 처음부터 다시 실행합니다.

In [ ]:
result = run_stream(legal_rag_agent, {"question": "대리인과 아파트 임대차 계약을 체결할 때 주의해야 할 점은 무엇인가요?"})
print(result.get("evaluation_report"))

In [ ]:
result = run_stream(legal_rag_agent, {"question": "개인정보 유출 시 기업이 취해야 할 법적 조치는 무엇인가요?"})
print(result.get("evaluation_report"))

## 6. Gradio 챗봇
* 웹 UI 에서는 노드 안에서 `input()` 을 쓸 수 없으므로 **중단점(`interrupt_before=["human_review"]`)** 을 사용합니다.
    1. 질문을 보내면 그래프가 평가까지 실행하고 `human_review` 직전에서 멈춥니다. → 답변과 평가 결과를 화면에 보여 줍니다.
    2. 사용자가 `y` / `n` 을 입력하면 `update_state` 로 `user_decision` 을 채우고 `invoke(None)` 으로 이어서 실행합니다.
    3. 승인(`y`)이면 최종 답변을, 거절(`n`)이면 다시 만든 답변과 평가를 보여 줍니다.
* 브라우저 세션마다 대화 상태가 분리되도록 `request.session_hash` 를 키로 세션을 관리합니다.
* 서버를 실행하면 노트북 셀이 계속 실행 상태가 됩니다. 마지막 셀의 `demo.close()` 로 종료하세요.

In [ ]:
import gradio as gr

# 중단점 + 체크포인터가 있는 그래프 (체크포인터가 있어야 멈춘 지점에서 재개 가능)
memory = MemorySaver()
legal_rag_agent = build_legal_rag_graph(review="breakpoint", checkpointer=memory)
show_graph(legal_rag_agent)

In [ ]:
import gradio as gr

# 예시 질문
EXAMPLE_QUESTIONS = [
    "사업장에서 CCTV를 설치할 때 주의해야 할 법적 사항은 무엇인가요?",
    "전월세 계약 갱신 요구권의 행사 기간과 조건은 어떻게 되나요?",
    "개인정보 유출 시 기업이 취해야 할 법적 조치는 무엇인가요?",
]
REVIEW_PROMPT = "이 답변을 승인하시겠습니까? (y/n): "

# 브라우저 세션별 상태: {session_id: {"thread_id": ..., "waiting": 승인 입력 대기 중인지}}
sessions: dict = {}


def format_review(values: dict, title: str) -> str:
    """중단점에서 얻은 상태값으로 '답변 + 평가 결과 + 승인 요청' 메시지를 만듭니다."""
    report = values.get("evaluation_report") or {}
    return (
        f"{title}\n{values.get('final_answer', '답변을 생성하지 못했습니다.')}\n\n"
        f"평가 결과:\n총점: {report.get('total_score', 0)}/60\n"
        f"{report.get('brief_evaluation', '평가 정보 없음')}\n\n{REVIEW_PROMPT}"
    )


def chat_fn(message: str, history: list, request: gr.Request = None) -> str:
    """Gradio 채팅 함수: 질문 → (중단) → 승인/거절 입력 → 최종 답변 순서로 진행합니다.

    history: Gradio 화면용 대화 기록 (상태는 체크포인터가 관리하므로 사용하지 않음)
    request: 브라우저 세션 식별용 (session_hash)
    """
    session_id = getattr(request, "session_hash", None) or "local"
    session = sessions.setdefault(session_id, {"thread_id": str(uuid.uuid4()), "waiting": False})
    config = {"configurable": {"thread_id": session["thread_id"]}}

    try:
        if not session["waiting"]:
            # 새 질문: human_review 직전까지 실행
            legal_rag_agent.invoke({"question": message}, config=config)
            state = legal_rag_agent.get_state(config)
            if not state.next:  # llm_fallback 경로처럼 승인 단계 없이 끝난 경우
                return state.values.get("final_answer", "응답을 생성하지 못했습니다.")
            session["waiting"] = True
            return format_review(state.values, "현재 답변:")

        # 승인 대기 중: y / n 만 허용
        decision = message.strip().lower()
        if decision not in ("y", "n"):
            return f"'y' 또는 'n' 으로 입력해 주세요. {REVIEW_PROMPT}"

        # 사람의 결정을 상태에 기록한 뒤 이어서 실행
        legal_rag_agent.update_state(config, {"user_decision": "approved" if decision == "y" else "rejected"})
        legal_rag_agent.invoke(None, config=config)
        state = legal_rag_agent.get_state(config)

        if decision == "y":
            session["waiting"] = False
            return state.values.get("final_answer", "최종 답변을 가져오지 못했습니다.")
        # 거절: 다시 만든 답변이 평가까지 끝나고 중단점에서 멈춘 상태 → 다시 승인 요청
        return format_review(state.values, "다시 생성한 답변:")

    except Exception as e:
        # 내부 오류 내용은 서버 로그에만 남기고 사용자에게는 일반 메시지를 보여 줌
        print(f"Error occurred: {e}")
        session["waiting"] = False
        return "죄송합니다. 응답을 생성하는 동안 오류가 발생했습니다. 다시 시도해 주세요."


demo = gr.ChatInterface(
    fn=chat_fn,
    title="생활법률 AI 어시스턴트",
    description="주택임대차보호법, 근로기준법, 개인정보보호법 관련 질문에 답변해 드립니다. 답변을 확인한 뒤 y/n 으로 승인해 주세요.",
    examples=EXAMPLE_QUESTIONS,
)

# Gradio 앱 실행 (기본 주소: http://127.0.0.1:7860)
demo.launch()

In [ ]:
# 데모 종료 (실행 중인 Gradio 서버 포트 해제)
demo.close()